<cell_type>markdown</cell_type># Energy Prices vs Inflation Analysis

What kind of relationship do gasoline, crude oil, and natural gas prices have with inflation — and how do their prices themselves behave through the year?

The notebook has two parts:

1. **Setup** — load the data, clean it, merge it, and plot the overall trends.
2. **Analysis 1: Energy prices vs. the inflation rate (with lead/lag).** Convert the CPI index into the year-over-year inflation rate, correlate it with each energy price, and check if energy prices *lead* inflation by some number of months.
3. **Analysis 2: Monthly seasonality of energy prices.** Are gas / oil / natural gas systematically more expensive in certain months of the year?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


### First I'll read the CSV files and store them in separate dataframes.

In [ ]:
gas = pd.read_csv("../data/raw/gas_prices.csv")
oil = pd.read_csv("../data/raw/crude_oil.csv")
natural = pd.read_csv("../data/raw/natural_gas.csv")
inflation = pd.read_csv("../data/raw/inflation.csv")

### Before doing analysis, I'll quickly check the first 5 rows of each dataset.

In [ ]:
print("Gas")
display(gas.head())

print("Oil")
display(oil.head())

print("Natural Gas")
display(natural.head())

print("Inflation")
display(inflation.head())

### Clean dates and keep only needed columns

Convert date columns into datetime format and simplify each dataframe.

In [ ]:
gas["period"] = pd.to_datetime(gas["period"])
oil["period"] = pd.to_datetime(oil["period"])
natural["date"] = pd.to_datetime(natural["date"])
inflation["date"] = pd.to_datetime(inflation["date"])

# FRED uses "." for missing values, so coerce to numeric
natural["value"] = pd.to_numeric(natural["value"], errors="coerce")
inflation["value"] = pd.to_numeric(inflation["value"], errors="coerce")

gas = gas[["period","value"]]
oil = oil[["period","value"]]
natural = natural[["date","value"]]
inflation = inflation[["date","value"]]

gas.columns = ["date","gas"]
oil.columns = ["date","oil"]
natural.columns = ["date","natural_gas"]
inflation.columns = ["date","cpi"]

### I'll join everything together by date so values appear side by side.

Because inflation data is monthly and the others are weekly, I'll make some changes so that all of them shows monthly data.

In [ ]:
# Convert all dates into month-year only
gas["date"] = gas["date"].dt.to_period("M")
oil["date"] = oil["date"].dt.to_period("M")
natural["date"] = natural["date"].dt.to_period("M")
inflation["date"] = inflation["date"].dt.to_period("M")

# Now I have 4-5 of values in the same month. Average all rows inside each of those months.
# (Natural gas and CPI are already monthly, so the average is just the single row.)
gas = gas.groupby("date")["gas"].mean().reset_index()
oil = oil.groupby("date")["oil"].mean().reset_index()
natural = natural.groupby("date")["natural_gas"].mean().reset_index()

df = gas.merge(oil, on="date")
df = df.merge(natural, on="date")
df = df.merge(inflation, on="date")

display(df.head())

### Plot Trends

I'll use low opacity so that overlapping lines remain visible.

In [ ]:
# Convert date if needed
if str(df["date"].dtype) == "period[M]":
    df["date"] = df["date"].dt.to_timestamp()

# Make numeric
cols = ["gas", "oil", "natural_gas", "cpi"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove missing values created during conversion
df = df.dropna()

# Create copy
scaled = df.copy()

# Normalize (each series starts at 1.0 so the shapes are comparable)
for col in cols:
    scaled[col] = scaled[col] / scaled[col].iloc[0]

# Plot
plt.figure(figsize=(14,6))

for col in cols:
    plt.plot(
        scaled["date"],
        scaled[col],
        label=col,
        alpha=0.6
    )

plt.xlabel("Date")
plt.ylabel("Normalized Value (start = 1.0)")
plt.title("Normalized Trends: Gas, Oil, Natural Gas, CPI")
plt.legend()

plt.show()

<cell_type>markdown</cell_type>## Analysis 1 — Energy Prices vs. the Inflation Rate (and Lead/Lag)

A quick note on the data: `inflation.csv` is actually the **CPI index level** (FRED series `CPIAUCSL`, with base period 1982–84 = 100). It is *not* the inflation rate. CPI grows almost monotonically, so a raw correlation of CPI vs energy prices doesn't really answer the question we care about. We need to convert the CPI into the year-over-year (YoY) inflation rate first.

Steps:

1. Convert CPI level → YoY inflation rate.
2. Plot energy prices alongside the inflation rate.
3. Correlate each energy price with the inflation rate.
4. Lead/lag check: shift the energy series back in time by 1, 2, ..., 12 months and see which lag has the strongest correlation. A best-lag of, say, 3 months would mean energy prices today match inflation three months from now (i.e. energy leads inflation by 3 months).

<cell_type>markdown</cell_type>### Step 1 — Build the YoY inflation rate

Year-over-year inflation = how much CPI changed compared to the same month last year, expressed as a percentage:

```
inflation_rate(t) = (CPI(t) / CPI(t - 12 months) - 1) * 100
```

The first 12 months will be NaN (because there is no "12 months ago" value yet), so we drop them.

In [ ]:
# Sort by date so the .shift(12) below uses the right "12 months ago" value
df = df.sort_values("date").reset_index(drop=True)

# YoY inflation rate, in percent
df["inflation_rate"] = (df["cpi"] / df["cpi"].shift(12) - 1) * 100

# Drop the first 12 rows (no 12-months-ago value to compare against)
df = df.dropna(subset=["inflation_rate"]).reset_index(drop=True)

display(df[["date", "cpi", "inflation_rate"]].head())

### Step 2 — Visualize inflation rate vs. energy prices

Twin y-axis plot: inflation rate on the right (%), energy prices on the left ($).

In [ ]:
fig, ax_left = plt.subplots(figsize=(14, 6))

# Energy prices on the left axis (oil is divided by 20 so it fits on the same scale)
ax_left.plot(df["date"], df["gas"], label="gas ($/gal)", alpha=0.7, color="tab:blue")
ax_left.plot(df["date"], df["oil"] / 20, label="oil ($/bbl ÷ 20)", alpha=0.7, color="tab:orange")
ax_left.plot(df["date"], df["natural_gas"], label="natural gas ($/MMBTU)", alpha=0.7, color="tab:green")
ax_left.set_xlabel("Date")
ax_left.set_ylabel("Energy price")
ax_left.legend(loc="upper left")

# Inflation rate on a separate right axis
ax_right = ax_left.twinx()
ax_right.plot(df["date"], df["inflation_rate"], label="YoY inflation %", color="black", linewidth=2)
ax_right.axhline(0, linestyle="--", linewidth=0.5, color="gray")
ax_right.set_ylabel("YoY inflation rate (%)")
ax_right.legend(loc="upper right")

plt.title("Energy prices vs. YoY inflation rate")
plt.show()

### Correlation against the inflation rate 

If the level-correlation in Analysis 1 was misleading, here is a version that answers the question.

In [ ]:
# How tightly do gas / oil / natural gas move with the YoY inflation rate?
correlations = df[["gas", "oil", "natural_gas", "inflation_rate"]].corr()
display(correlations)

print("\nCorrelation of each energy price with the YoY inflation rate:")
print(correlations["inflation_rate"].drop("inflation_rate").sort_values(ascending=False))

### Lead/lag analysis

The correlation above answers "do they move together this month?" But energy is an **input cost** — a gas-price spike might take a few months to filter through transportation, packaging, food, and into the overall CPI basket.

For each lag `k = 0, 1, 2, ..., 12` months, we shift the energy series back by `k` months and correlate with today's inflation rate. The lag with the highest correlation is the one where energy "leads" inflation by that many months.

In [ ]:
# Step 1: convert each energy series to its own YoY % change.
# (We compare changes-to-changes, not levels-to-levels.)
df["gas_yoy"]         = (df["gas"]         / df["gas"].shift(12)         - 1) * 100
df["oil_yoy"]         = (df["oil"]         / df["oil"].shift(12)         - 1) * 100
df["natural_gas_yoy"] = (df["natural_gas"] / df["natural_gas"].shift(12) - 1) * 100

# Step 2: for each lag k = 0..12 months, correlate "energy YoY shifted back by k months"
# with "today's inflation rate". We store the result in a simple table.
energy_yoy_cols = ["gas_yoy", "oil_yoy", "natural_gas_yoy"]
lag_table = pd.DataFrame(index=range(0, 13), columns=energy_yoy_cols, dtype=float)
lag_table.index.name = "lag_months"

for col in energy_yoy_cols:
    for k in lag_table.index:
        lag_table.loc[k, col] = df[col].shift(k).corr(df["inflation_rate"])

display(lag_table.round(3))

print("\nBest lag for each energy series (the month where the correlation is highest):")
for col in energy_yoy_cols:
    best_lag = lag_table[col].idxmax()
    best_corr = lag_table[col].max()
    print(f"  {col:18s}  lag = {best_lag:2d} months   corr = {best_corr:+.3f}")

In [ ]:
plt.figure(figsize=(12, 5))

for col in energy_yoy_cols:
    plt.plot(lag_table.index, lag_table[col], marker="o", label=col)

plt.axhline(0, color="gray", linewidth=0.5)
plt.xlabel("Lag in months (energy series shifted back by this many months)")
plt.ylabel("Correlation with YoY inflation")
plt.title("Lead/lag: YoY energy price change vs. YoY inflation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

<cell_type>markdown</cell_type>### My Analysis (Analysis 1)

**Step 3 — straightforward correlations with the YoY inflation rate**

| Variable        | Corr. with YoY inflation |
|-----------------|--------------------------|
| oil             | **+0.69** |
| gas (retail)    | **+0.63** |
| natural gas     | **-0.22** |

- **Oil and retail gasoline both move with inflation in a strong, positive way.** Crude oil is the strongest single co-mover (+0.69). This matches the textbook story: oil is an upstream input cost, so when oil rises faster, downstream prices in the CPI basket rise faster too.
- **Henry Hub natural gas does *not* track inflation (-0.22).** US natural gas spot prices are dominated by domestic factors (storage, weather, winter heating demand, shale supply) rather than the broad global inflation cycle. So "all energy moves with inflation" is mostly true for *oil* (and gasoline, which is a derivative of oil) but not for natural gas.

**Step 4 — lead/lag**

Best (highest-correlation) lag for each series:

| Variable     | Best lag | Correlation |
|--------------|----------|-------------|
| gas          | 0 months | 0.91 |
| oil          | 0 months | 0.88 |
| natural gas  | 12 months | 0.96 |

- For gas and oil the **strongest correlation is at lag 0** — their YoY changes move **contemporaneously** with YoY inflation, not ahead of it. The textbook story would predict a positive lead (oil moves first, CPI catches up a few months later), and the lag-1 correlation is still high (≈0.75 for gas, 0.68 for oil), but the present-day correlation just edges it out in this dataset.
- The natural gas "best lag = 12 months" result is **almost certainly a small-sample artifact** rather than a real lead. After YoY-differencing inflation, YoY-differencing natural gas, and then shifting back 12 months we have thrown out ~36 of the ~130 monthly observations, and the remaining sample is dominated by the single 2021–22 inflation arc. The lag-12 row also shows oil at -1.00, which is the same artifact in reverse. Treat anything past lag ≈ 6 as unreliable here.
- The honest contemporaneous reading is: **oil and retail gasoline move tightly with inflation in real time; natural gas does not.**

**Caveats**

- 2015–2026 is one specific macro window (deflationary 2015, COVID shock 2020, 2021–22 inflation surge, 2023–24 disinflation). One regime, one shock — correlations of this size on a sample this short shouldn't be taken as stable. Different decades (e.g. the 1970s/80s oil shocks) would not necessarily give the same lead times.
- We didn't control for monetary policy, dollar strength, or supply-chain shocks, all of which affect energy prices and CPI simultaneously. Correlation here is not causation.

## Analysis 2 — Monthly Seasonality of Energy Prices

Different question this time: forget inflation for a moment. **Do energy prices have a regular seasonal pattern through the year?**

Intuitively you might guess:

- **Gasoline** is most expensive in summer (people drive more for vacation, refineries do maintenance).
- **Natural gas** is most expensive in winter (heating demand).
- **Crude oil** doesn't have an obvious seasonal hook.

We can check this directly: for each month of the year (Jan = 1, …, Dec = 12), average the price across all years in the data, then look for a U-shape, hump, or flat line.

### Step 1 — Handle the price trend first

Prices have grown a lot over 2015–2026 (especially in 2022). If we just averaged prices for every January across years, the average would be pulled up by recent years and wouldn't isolate the *seasonal* signal — it would mix seasonality with the long-term trend.

Fix: for each year, divide every monthly price by **that year's own average**. The result is a ratio: 1.05 means "5% above average for this year", 0.95 means "5% below". Now we can compare months across years on the same footing.

In [ ]:
# Add year and month-of-year columns
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month

# For each year, compute that year's mean price for each energy series
yearly_means = df.groupby("year")[["gas", "oil", "natural_gas"]].transform("mean")

# Divide each monthly price by its year's mean -> gives a ratio (1.0 = exactly the year's average)
seasonal = df[["year", "month"]].copy()
seasonal["gas"]         = df["gas"]         / yearly_means["gas"]
seasonal["oil"]         = df["oil"]         / yearly_means["oil"]
seasonal["natural_gas"] = df["natural_gas"] / yearly_means["natural_gas"]

display(seasonal.head())

### Step 2 — Average each month across all years

Now group by month-of-year. The resulting table has 12 rows (one per month) and shows the typical seasonal "shape" of each energy series, with the long-term trend removed.

In [ ]:
monthly_pattern = seasonal.groupby("month")[["gas", "oil", "natural_gas"]].mean()

# Show as percentages above/below 1.0 for easier reading
display((monthly_pattern * 100 - 100).round(2).rename(columns=lambda c: f"{c} (% vs year avg)"))

### Step 3 — Plot the seasonal shape

A line plot makes the shape obvious. The dashed line at 1.0 is the yearly average — anything above means "more expensive than typical for that year", anything below means "cheaper than typical".

In [ ]:
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

plt.figure(figsize=(11, 5))
plt.plot(monthly_pattern.index, monthly_pattern["gas"],         marker="o", label="gas")
plt.plot(monthly_pattern.index, monthly_pattern["oil"],         marker="o", label="oil")
plt.plot(monthly_pattern.index, monthly_pattern["natural_gas"], marker="o", label="natural gas")

plt.axhline(1.0, color="gray", linestyle="--", linewidth=0.7)
plt.xticks(range(1, 13), month_labels)
plt.xlabel("Month of year")
plt.ylabel("Price relative to that year's average  (1.0 = average)")
plt.title("Seasonal price pattern (averaged across 2015–2026)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Step 4 — Quantify the seasonal "size"

How big is the seasonal effect for each series? A simple measure: difference between the most-expensive month and the cheapest month, in percent.

In [ ]:
for col in ["gas", "oil", "natural_gas"]:
    high_month = monthly_pattern[col].idxmax()
    low_month  = monthly_pattern[col].idxmin()
    spread_pct = (monthly_pattern[col].max() / monthly_pattern[col].min() - 1) * 100

    print(
        f"{col:12s}  most expensive: {month_labels[high_month-1]}"
        f"   cheapest: {month_labels[low_month-1]}"
        f"   peak-to-trough: {spread_pct:.1f}%"
    )

<cell_type>markdown</cell_type>### My Analysis (Analysis 2)

| Series        | Most expensive month | Cheapest month | Peak-to-trough |
|---------------|----------------------|----------------|----------------|
| gas (retail)  | **April**            | January        | ~21%           |
| oil (WTI)     | **April**            | December       | ~25%           |
| natural gas   | **January**          | April          | ~89%           |

A few things stand out:

- **Gasoline and oil have the same seasonal shape — and it peaks in April, not July.** Gas climbs sharply from January (-8%) to April (+11%), stays elevated through summer (May-September are 1-6% above the yearly average), then falls back into late winter. Oil follows the same arc (peak +12% in April, trough -10% in December). This matches the well-known *summer driving season* effect, which begins each spring when US refineries switch from cheaper winter-blend to more expensive summer-blend gasoline and demand from drivers rises. Crude oil tracks the same pattern because much of US crude demand goes into making gasoline. So a tempting intuition ("oil has no obvious seasonal hook") would be wrong — oil inherits gasoline's seasonality, just slightly weaker.

- **Natural gas seasonality is huge and runs in the opposite direction.** January sits ~47% above the yearly average, December ~28% above, while April sits ~22% below. Peak-to-trough is about 89% — roughly four times larger than the gasoline/oil swing. The reason is obvious: natural gas is the primary US heating fuel, and demand spikes when it gets cold. There is no equivalent "summer demand" because most A/C runs on electricity (which can be drawn from many sources, not just gas).

- **Practical takeaway.** Across all three series, **April is the cheapest month to buy gas/oil-derived products only if you ignore the small dip into mid-winter**, and is decisively the cheapest month for natural gas. If you wanted to time refilling a home heating-oil or natural-gas tank, April is the trough.

**Caveats**

- The January natural-gas figure (+47% vs. yearly average) is inflated somewhat by one or two extreme winters in the dataset (notably the Jan 2026 spike to $7.72/MMBTU). The general "winter is much more expensive than spring" shape is robust, but the exact magnitude is sensitive to those outliers.
- 11 years of monthly data is not a lot — for a single month-of-year, we're averaging only ~11 observations. Don't read too much into a single month's value; trust the overall U/inverted-U *shape* more than the individual numbers.
- The trend-removal step (dividing each month by its year's mean) handles the level drift over 2015–2026, but it does not handle structural changes (e.g. the US shale boom permanently lowered natural gas levels; COVID temporarily flattened gasoline seasonality in 2020). A more sophisticated analysis (e.g. STL decomposition) would tease those apart, but the simple groupby is enough to see the main pattern.